# 02 — Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

CLEAN_PATH = Path('../data/clean/clean_game_nba.csv')
TEAM_GAMES_PATH = Path('../data/modelling/team_games.csv')
MODEL_DATA_PATH = Path('../data/modelling/model_data.csv')

# Load
games = pd.read_csv(CLEAN_PATH)
games['game_date'] = pd.to_datetime(games['game_date'], errors='coerce')

# Split home/away blocks
home_cols = [c for c in games.columns if c.endswith('_home')]
away_cols = [c for c in games.columns if c.endswith('_away')]

home_df = games[['game_id','season_id','game_date'] + home_cols].copy()
home_df['team_id'] = games['team_id_home']
home_df['team_name'] = games['team_name_home']
home_df['team_win'] = games['home_win']
home_df['home_away_flag'] = 'home'

away_df = games[['game_id','season_id','game_date'] + away_cols].copy()
away_df['team_id'] = games['team_id_away']
away_df['team_name'] = games['team_name_away']
away_df['team_win'] = 1 - games['home_win']
away_df['home_away_flag'] = 'away'

# Prefix stats uniformly
home_df = home_df.rename(columns={c: 'stat_' + c.replace('_home','') for c in home_cols})
away_df = away_df.rename(columns={c: 'stat_' + c.replace('_away','') for c in away_cols})

# Align common
common_cols = sorted(set(home_df.columns) & set(away_df.columns))
home_df = home_df[common_cols]
away_df = away_df[common_cols]

# Stack
team_games = pd.concat([home_df, away_df], ignore_index=True)
team_games = team_games.sort_values(['team_id','game_date']).reset_index(drop=True)

# Rolling features
stat_cols = [c for c in team_games.columns if c.startswith('stat_') and pd.api.types.is_numeric_dtype(team_games[c])]
windows = [3,5,10]
for stat in stat_cols:
    for w in windows:
        team_games[f'{stat}_roll_{w}'] = (
            team_games.groupby('team_id')[stat]
                      .rolling(w)
                      .mean()
                      .shift(1)
                      .reset_index(level=0, drop=True)
        )

# Win streaks (last N wins)
streak_windows = [1,3,5,10]
for w in streak_windows:
    team_games[f'win_streak_{w}'] = (
        team_games.groupby('team_id')['team_win']
                  .rolling(w)
                  .sum()
                  .shift(1)
                  .reset_index(level=0, drop=True)
    )

# Rest features
team_games['prev_game_date'] = team_games.groupby('team_id')['game_date'].shift(1)
team_games['rest_days'] = (team_games['game_date'] - team_games['prev_game_date']).dt.days
team_games['is_b2b'] = (team_games['rest_days'] <= 1).astype(int)
team_games['rest_3plus'] = (team_games['rest_days'] >= 3).astype(int)

# Season progress
team_games = team_games.sort_values(['team_id','season_id','game_date'])
team_games['season_win_pct'] = (
    team_games.groupby(['team_id','season_id'])['team_win']
              .expanding()
              .mean()
              .shift(1)
              .reset_index(level=[0,1], drop=True)
)
team_games['season_wins'] = (
    team_games.groupby(['team_id','season_id'])['team_win']
              .cumsum()
              .shift(1)
)
team_games['season_game_number'] = team_games.groupby(['team_id','season_id']).cumcount()

# Persist team_games
TEAM_GAMES_PATH.parent.mkdir(parents=True, exist_ok=True)
team_games.to_csv(TEAM_GAMES_PATH, index=False)
print(f"Saved {TEAM_GAMES_PATH}")

# Merge to modelling frame with home/away and diffs
home_feats = team_games[team_games['home_away_flag']=='home'].copy().add_prefix('home_')
away_feats = team_games[team_games['home_away_flag']=='away'].copy().add_prefix('away_')

model_data = games[['game_id','home_win','team_id_home','team_id_away']].copy()
model_data = model_data.merge(home_feats, left_on='game_id', right_on='home_game_id', how='left')\
                       .merge(away_feats, left_on='game_id', right_on='away_game_id', how='left')

# Drop id/name text columns that could leak / blow up features
model_data = model_data.drop(columns=['home_game_id','away_game_id','team_id_home','team_id_away',
                                     'home_team_id','away_team_id'], errors='ignore')

# Numeric-only diff columns
numeric_home = [c for c in model_data.columns if c.startswith('home_stat_') and pd.api.types.is_numeric_dtype(model_data[c])]
numeric_away = [c for c in model_data.columns if c.startswith('away_stat_') and pd.api.types.is_numeric_dtype(model_data[c])]

for h in numeric_home:
    base = h.replace('home_stat_','')
    a = 'away_stat_' + base
    if a in numeric_away:
        model_data['diff_' + base] = pd.to_numeric(model_data[h], errors='coerce') - pd.to_numeric(model_data[a], errors='coerce')

# Fill NA safely (post-rolling)
model_data = model_data.fillna(0)

# Remove residual text cols
drop_text = [c for c in model_data.columns if 'team_id' in c or 'team_name' in c or 'matchup' in c]
model_data = model_data.drop(columns=drop_text, errors='ignore')

# Persist
MODEL_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
model_data.to_csv(MODEL_DATA_PATH, index=False)
print(f"Saved {MODEL_DATA_PATH}; shape={model_data.shape}")
